In [1]:
%%capture
%cd ../
import os
os.environ["PYTHONHASHSEED"] = "42"
%load_ext autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from config.env in the repository root
repo_root = Path().resolve().parents[1]  # Navigate up from notebooks -> training -> src to repo root
load_dotenv(repo_root / 'config.env')

# === Select data version here ===
DATA_VERSION = "v2"  # Change to "v2", "v3", etc. as needed
# =================================

TRAINING_DATA_PATH = os.environ.get(f'TRAINING_DATA_{DATA_VERSION.upper()}')
if not TRAINING_DATA_PATH:
    raise ValueError(f'TRAINING_DATA_{DATA_VERSION.upper()} not set. Check config.env.')

In [3]:
import logging

import pandas as pd
from datetime import datetime

from ml_common.summary import get_label_distribution
from ml_common.util import load_pickle, save_pickle

from acu.evaluation import evaluate_valid, evaluate_test, predict, compute_threshold
from acu.pipeline import PrepACUData, feature_summary
from acu.training import set_seed, train_models, tune_params

set_seed()

pd.set_option('display.max_rows', 150)
pd.set_option('display.max_columns', 100)

logging.basicConfig(
    level=logging.INFO, 
    format='%(levelname)s:%(message)s', 
)

In [4]:
df = pd.read_parquet(TRAINING_DATA_PATH)
if "target_ED_date" not in df.columns:
    dates_path = Path(TRAINING_DATA_PATH).parent / "clinic_centered_dates.parquet"
    dates_df = pd.read_parquet(dates_path)
    df = df.merge(dates_df[["mrn", "assessment_date", "target_ED_date", "treatment_date"]],
                  on=["mrn", "assessment_date"], how="left")

In [5]:
prep = PrepACUData()
df = prep.preprocess(df, start_date="2006-01-01", end_date="2024-09-03")

01:30:47 INFO:Removing 22716 patients and 148507 sessions not from GI department
01:30:47 INFO:Removing 13 patients and 265 sessions before 2006-01-01 and after 2024-09-03
01:30:47 INFO:Removing 16 patients and 110 sessions in which patient had an ED visit on or before their treatment date.
01:30:47 INFO:Removing 0 patients and 12 sessions not first of a given week
01:30:47 INFO:Dropping the following 0 features for missingness over 80%: []
01:30:48 INFO:Removing 0 patients and 0 sessions with at least 80 percent of features missing
01:30:48 INFO:Reassigning the following 14 indicators with less than 6 patients as other: ['cancer_site_C02', 'cancer_site_C06', 'cancer_site_C07', 'cancer_site_C09', 'cancer_site_C12', 'cancer_site_C31', 'cancer_site_C37', 'cancer_site_C47', 'cancer_site_C48', 'cancer_site_C51', 'cancer_site_C53', 'cancer_site_C60', 'cancer_site_C65', 'cancer_site_C76']
01:30:48 INFO:No rare categories for morphology was found


In [6]:
# need to specify train test split date
X, Y, metainfo = prep.prepare('2021-12-31', df, n_folds=5) # n_folds=3
df = df.loc[X.index]

01:30:52 INFO:Removing 0 patients and 1778 sessions that occured after 2021-12-31 in the development cohort
01:30:52 INFO:One-hot encoding training data
01:30:52 INFO:Reassigning the following 15 indicators with less than 6 patients as other: ['regimen_GI OP FOLFOX + Nivolumab', 'regimen_GI OP FOLFOX + Pembolizumab', 'regimen_GI Pembrolizumab Q3W', 'regimen_GI Raltitrexed', 'regimen_GI-AVEX', 'regimen_GI-CISCAP+TRASLOAD(BS)', 'regimen_GI-DOCEQ3W', 'regimen_GI-EOX', 'regimen_GI-FOLFOX+PANITUMUMAB', 'regimen_GI-GEM+OXALI (BILIARY)', 'regimen_GI-GEM7WEEK+ERLOTINIB', 'regimen_GI-GEMFU (BILIARY)', 'regimen_GI-IRINOCISP', 'regimen_GI-OXALI', 'regimen_GI-XELOX+BEVACIZUMAB']
01:30:52 INFO:No rare categories for intent was found
01:30:52 INFO:One-hot encoding testing data
01:30:52 INFO:Reassigning the following regimen indicator columns that did not exist in train set as other:
regimen_GI Cetuximab + Encorafenib                                  14
regimen_GI Durvalumab                          

In [7]:
# reset X, Y, metainfo, df indices
X = X.reset_index(drop=True)
Y = Y.reset_index(drop=True)
metainfo = metainfo.reset_index(drop=True)
df = df.reset_index(drop=True)

# split into train and test
train_mask, test_mask = metainfo['split'] == 'Train', metainfo['split'] == 'Test'
X_train, X_test = X[train_mask], X[test_mask]
Y_train, Y_test = Y[train_mask], Y[test_mask]
metainfo_train, metainfo_test = metainfo[train_mask], metainfo[test_mask]
df_train, df_test = df[train_mask], df[test_mask]

In [ ]:
# make sure there is no patient leakage
set(metainfo_train['mrn'].unique()) & set(metainfo_test['mrn'].unique())

set()

In [9]:
count = pd.DataFrame({
    'Number of sessions': metainfo.groupby('split').apply(len, include_groups=False), 
    'Number of patients': metainfo.groupby('split')['mrn'].nunique()}
).T
count['Total'] = count.sum(axis=1)
print(f'\n{count.to_string()}')


split               Test  Train  Total
Number of sessions  3945  22933  26878
Number of patients   744   4453   5197


In [10]:
mask = X['cycle_number'] == X_train['cycle_number'].min()
count = pd.DataFrame({
    'Number of sessions': metainfo[mask].groupby('split').apply(len, include_groups=False), 
    'Number of patients': metainfo[mask].groupby('split')['mrn'].nunique()}
).T
count['Total'] = count.sum(axis=1)
print(f'\n{count.to_string()}')


split               Test  Train  Total
Number of sessions   698   4068   4766
Number of patients   529   3218   3747


In [18]:
# Feature Characteristics
x = prep.ohe.encode(df.loc[X_train.index].copy(), verbose=False) # get original (non-normalized, non-imputed) data one-hot encoded
x = x[[col for col in x.columns if not (col in metainfo.columns or col.startswith('target'))]]
feature_summary(x, save_path=f'./deployment/feature_summary_ED_clinic_anchored.csv').sample(10, random_state=42)

,Features,Group,Mean (SD),Missingness (%)
28,Monocyte (x10e9/L),Laboratory,0.595 (0.326),27.6
13,Calcium (mmol/L),Laboratory,2.350 (0.128),13.6
70,"Topography ICD-0-3 C54, Corpus uteri",Cancer,0.002 (0.044),0.0
2,Height (cm),Demographic,167.503 (9.823),0.0
105,"Regimen GI-FUFA+RT CYC 3,4",Treatment,0.001 (0.023),0.0
29,Neutrophil (x10e9/L),Laboratory,3.854 (2.814),25.9
51,Days Since Starting Treatment,Treatment,147.077 (266.560),0.0
111,Regimen GI-GEMCAP,Treatment,0.023 (0.150),0.0
88,Regimen GI-CISPFU ANAL,Treatment,0.001 (0.030),0.0
77,"Topography ICD-0-3 C74, Adrenal Gland",Cancer,0.001 (0.031),0.0


# Model Training

In [11]:
# LGBM does not like non alphanumeric characters (except for _)
for char in ['(', ')', '+', '-', '/', ',']: 
    X_train.columns = X_train.columns.str.replace(char, '_')
    X_test.columns = X_test.columns.str.replace(char, '_')
    X.columns = X.columns.str.replace(char, '_')

In [12]:
%%capture
# Hyperparameter tuning
# TODO: try greater kappa for greater exploration
algs = ['LASSO', 'RF', 'Ridge', 'XGB', 'LGBM']
best_params = {}
for alg in algs:
    best_params[alg] = tune_params(alg, X_train, Y_train, metainfo_train, log_dir='./deployment/logs/bayes_opt')
os.makedirs('./deployment/models', exist_ok=True)
save_pickle(best_params, './deployment/models', f'best_params_clinic_anchored')

In [13]:
best_params = load_pickle('./deployment/models', f'best_params_clinic_anchored')
models = train_models(X_train, Y_train, metainfo_train, best_params) # NOTE: Number of CV folds = 3

/Users/wayne/miniconda3/envs/model_deployer/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/wayne/miniconda3/envs/model_deployer/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/wayne/miniconda3/envs/model_deployer/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/wayne/miniconda3/envs/model_deployer/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

## Model Selection
Select final model based on the average performance across the validation folds

In [14]:
evaluate_valid(models, X_train, Y_train, metainfo_train)

Ridge               LASSO                 XGB                LGBM  \
           AUPRC     AUROC     AUPRC     AUROC     AUPRC     AUROC     AUPRC   
ED_30d  0.223616  0.785214  0.224716  0.785317  0.938835  0.995362  0.201681   

                        RF            
           AUROC     AUPRC     AUROC  
ED_30d  0.785407  0.338411  0.820635

## Evaluate Model

In [15]:
pd.concat([evaluate_test(model, X_test, Y_test) for alg, model in models.items()], keys=models.keys()).T

Ridge                         LASSO                         XGB  \
           AUPRC     AUROC     Brier     AUPRC     AUROC    Brier    AUPRC   
ED_30d  0.197927  0.678115  0.076202  0.200986  0.680405  0.07602  0.16309   

                                LGBM                            RF            \
           AUROC     Brier     AUPRC     AUROC     Brier     AUPRC     AUROC   
ED_30d  0.654122  0.077453  0.183031  0.696702  0.076089  0.223346  0.712079   

                  
           Brier  
ED_30d  0.074496

In [16]:
# first cycle only
mask = X_test['cycle_number'] == X_train['cycle_number'].min()
pd.concat([evaluate_test(model, X_test[mask], Y_test[mask]) for alg, model in models.items()], keys=models.keys()).T

Ridge                        LASSO                           XGB  \
          AUPRC    AUROC     Brier     AUPRC     AUROC     Brier     AUPRC   
ED_30d  0.34006  0.70004  0.127386  0.326821  0.700893  0.127434  0.279608   

                                LGBM                           RF            \
           AUROC     Brier     AUPRC     AUROC    Brier     AUPRC     AUROC   
ED_30d  0.654833  0.132953  0.286136  0.674488  0.13005  0.347683  0.726879   

                  
           Brier  
ED_30d  0.122558

In [17]:
chosen_model = 'RF'
model = models[f'{chosen_model}']
save_pickle(model['ED_30d'], './deployment/models', f'{chosen_model}_ED_visit_clinic_anchored')
X.to_parquet(f'./deployment/models/X_clinic_anchored.parquet.gzip')
Y.to_parquet(f'./deployment/models/Y_clinic_anchored.parquet.gzip')
save_pickle(prep, './deployment', f'prep_ED_visit_clinic_anchored')